In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# Mendapatkan direktori aktif notebook (asumsi dijalankan di dalam /src/pipelines/training atau sejenisnya)
script_dir = os.getcwd() 

# [MODIFIKASI PENTING]: Buat direktori /final_comparison baru jika belum ada
FINAL_COMP_DIR = os.path.join(script_dir, 'final_comparison')
os.makedirs(FINAL_COMP_DIR, exist_ok=True)
print(f"📁 Folder output untuk CSV disiapkan di: {FINAL_COMP_DIR}\n")

# List subjek dari SUB1 sampai SUB12
subjects = [f"SUB{i}" for i in range(1, 13)]

# PERBAIKAN: Menambahkan 'WER' tepat setelah 'cer' pada daftar metrik
metrics = ['cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore']

print("Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...\n")

# ============================================================================
# 1. KONFIGURASI EKSPERIMEN (SINGLE & ALL SUBJECTS)
# ============================================================================
# Kita gabungkan konfigurasinya agar lebih rapi. 
# Pastikan template merujuk ke file yang memiliki prefix "complete_metrics_"
experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_{subject}_eq_3_0_log-mel_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_{subject}_eq_3_0_logmel_test_predictions_10_1_IndoGPT.csv"
    },
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_all_eq_3_0_log-mel_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_all_eq_3_0_logmel_test_predictions_10_1_IndoGPT.csv"
    }
]

# Dictionary untuk menampung baris data setiap tabel metrik
tables_data = {m: [] for m in metrics}

# List untuk menampung baris data pada tabel Ringkasan Global (Grand Summary)
summary_rows = []

# ============================================================================
# 2. PROSES EKSTRAKSI DATA
# ============================================================================

for exp in experiments:
    # Siapkan kerangka baris untuk tabel ringkasan global
    summary_row = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    # Siapkan kerangka baris untuk masing-masing tabel metrik detail
    exp_metric_rows = {m: {'Decoder': exp['Decoder'], 'Train Data': exp['Train Data']} for m in metrics}
    
    # Wadah untuk menampung nilai mentah (raw values) agar bisa dihitung Micro-Average
    raw_values = {m: [] for m in metrics}
    
    # --- JIKA EKSPERIMEN SINGLE SUBJECT ---
    if exp['type'] == 'single':
        for subject in subjects:
            filename = exp['template'].format(subject=subject)
            filepath = os.path.join(script_dir, filename)
            
            if os.path.exists(filepath):
                try:
                    df = pd.read_csv(filepath)
                    for m in metrics:
                        if m in df.columns:
                            exp_metric_rows[m][subject] = round(df[m].mean(), 4)
                            raw_values[m].extend(df[m].dropna().tolist())
                        else:
                            exp_metric_rows[m][subject] = np.nan
                except Exception as e:
                    print(f"  [ERROR] Gagal membaca {filename}: {e}")
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
            else:
                for m in metrics: exp_metric_rows[m][subject] = np.nan
                
    # --- JIKA EKSPERIMEN ALL SUBJECTS ---
    elif exp['type'] == 'all':
        filename = exp['template']
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_all = pd.read_csv(filepath)
                for subject in subjects:
                    sub_df = df_all[df_all['subject'] == subject]
                    if len(sub_df) > 0:
                        for m in metrics:
                            if m in sub_df.columns:
                                exp_metric_rows[m][subject] = round(sub_df[m].mean(), 4)
                                raw_values[m].extend(sub_df[m].dropna().tolist())
                            else:
                                exp_metric_rows[m][subject] = np.nan
                    else:
                        for m in metrics: exp_metric_rows[m][subject] = np.nan
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                for subject in subjects:
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
        else:
            print(f"  ⚠️ [INFO] File {filename} belum ditemukan.")
            for subject in subjects:
                for m in metrics: exp_metric_rows[m][subject] = np.nan

    # --- HITUNG RATA-RATA GLOBAL (MICRO-AVERAGE) UNTUK SETIAP METRIK ---
    for m in metrics:
        if raw_values[m]:
            global_avg = round(np.mean(raw_values[m]), 4)
        else:
            global_avg = np.nan
            
        exp_metric_rows[m]['Rata-rata Global'] = global_avg
        tables_data[m].append(exp_metric_rows[m])
        
        # Masukkan rata-rata global ini ke dalam tabel Ringkasan
        summary_row[m] = global_avg
        
    summary_rows.append(summary_row)

# ============================================================================
# 3. MENAMPILKAN TABEL DETAIL (PER METRIK) & MENYIMPAN KE CSV
# ============================================================================
column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']

for m in metrics:
    df_metric = pd.DataFrame(tables_data[m])
    # Urutkan kolom sesuai format
    df_metric = df_metric[[col for col in column_order if col in df_metric.columns]]
    
    print("=" * 110)
    print(f"TABEL PERBANDINGAN PERFORMA: {m.upper()}")
    print("=" * 110)
    display(df_metric)
    
    # [MODIFIKASI PENTING]: Menyimpan tabel metrik individual ke CSV
    metric_csv_path = os.path.join(FINAL_COMP_DIR, f"log-mel_normal_{m}.csv")
    df_metric.to_csv(metric_csv_path, index=False)
    print(f"✅ Tersimpan: {metric_csv_path}\n")

# ============================================================================
# 4. MEMBUAT, MENAMPILKAN, & MENYIMPAN TABEL GRAND SUMMARY
# ============================================================================
df_summary_global = pd.DataFrame(summary_rows)

# Urutkan kolom tabel ringkasan
summary_col_order = ['Decoder', 'Train Data'] + metrics
df_summary_global = df_summary_global[[col for col in summary_col_order if col in df_summary_global.columns]]

print("*" * 110)
print("🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆")
print("*" * 110)
display(df_summary_global)

# [MODIFIKASI PENTING]: Menyimpan tabel Grand Summary ke CSV
summary_csv_path = os.path.join(FINAL_COMP_DIR, "log-mel_normal_grand_summary.csv")
df_summary_global.to_csv(summary_csv_path, index=False)
print(f"✅ Tersimpan: {summary_csv_path}\n")

# ============================================================================
# EVALUASI OOV (OUT-OF-VOCABULARY)
# ============================================================================

# ==========================================
# KONFIGURASI PATH & PARAMETER
# ==========================================
CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

# List untuk menyimpan hasil OOV agar bisa dijadikan tabel
oov_summary_list = []

print("\n\n" + "=" * 110)
print("MEMULAI ANALISIS OOV KARAKTER...")
print("=" * 110)

for subject in subjects:
    # Definisikan nama file yang sudah displit
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    # Periksa apakah ketiga file untuk subjek ini benar-benar ada
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        # 1. Load Data
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # 2. Analisis OOV Karakter (Train vs Test)
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        # Cari karakter yang ada di Test tapi tidak ada di Train
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        # 3. Simpan metrik ke dalam list untuk tabel
        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

# ==========================================
# MENAMPILKAN & MENYIMPAN TABEL SUMMARY OOV
# ==========================================
if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    # Menampilkan tabel (gunakan display agar format tabel interaktif Jupyter keluar)
    print("\n" + "=" * 110)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 110)
    display(df_summary)
    
    # [MODIFIKASI PENTING]: Menyimpan tabel OOV ke CSV
    oov_csv_path = os.path.join(FINAL_COMP_DIR, "oov_tokenizer_char.csv")
    df_summary.to_csv(oov_csv_path, index=False)
    print(f"✅ Tersimpan: {oov_csv_path}\n")
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

📁 Folder output untuk CSV disiapkan di: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison

Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen Log-Mel Spectrogram...

TABEL PERBANDINGAN PERFORMA: CER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,76.8076,72.6506,76.7674,75.9840,78.0749,77.0432,78.3345,73.3874,77.7507,77.0753,75.6148,77.5256,76.4196
1,IndoGPT,Single,76.4745,78.3908,72.4532,76.9184,74.2505,91.6032,74.4060,70.8243,72.2037,75.9891,73.7744,82.3000,76.4788
2,LSTM,All,75.5622,75.5066,80.6789,84.2840,74.1315,76.7655,73.3556,71.2251,74.1591,74.0902,74.7761,71.9370,75.6756
3,IndoGPT,All,74.1153,70.2708,80.8467,75.5702,72.6871,74.5613,74.2522,71.3450,79.0202,70.4363,73.7568,78.8690,74.3066


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_normal_cer.csv

TABEL PERBANDINGAN PERFORMA: WER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,99.1667,96.6667,97.5000,95.0833,98.3333,101.0000,100.0000,94.2857,98.9474,105.7018,97.6471,100.0000,98.7516
1,IndoGPT,Single,111.8690,98.0000,98.3333,105.9167,100.8333,118.7500,99.9940,96.4881,99.6491,101.7544,97.9832,113.7500,103.5947
2,LSTM,All,101.6667,96.0000,107.5000,113.7500,102.5000,104.1667,98.3333,94.2500,100.7018,101.2657,100.9244,101.2500,101.9123
3,IndoGPT,All,96.9881,95.0000,103.0000,101.0833,96.5556,100.6667,101.2857,94.5357,105.0877,94.0351,98.1793,108.3333,99.0606


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_normal_WER.csv

TABEL PERBANDINGAN PERFORMA: BLEU-1


...

✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_normal_BertScore.csv

**************************************************************************************************************
🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆
**************************************************************************************************************


,Decoder,Train Data,cer,WER,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1-P,ROUGE-1-R,ROUGE-1-F,BertScore
0,LSTM,Single,76.4196,98.7516,2.6385,1.0037,0.8742,0.8370,5.3263,3.2124,3.9284,44.3036
1,IndoGPT,Single,76.4788,103.5947,4.6354,2.0547,1.5641,1.3983,7.5674,5.5033,6.1593,49.3901
2,LSTM,All,75.6756,101.9123,3.5343,1.6747,1.2624,1.1286,6.2081,4.0552,4.7687,46.8744
3,IndoGPT,All,74.3066,99.0606,6.1297,2.8017,2.2533,2.0908,8.5979,6.5359,7.3581,53.5146


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\log-mel_normal_grand_summary.csv



MEMULAI ANALISIS OOV KARAKTER...

RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\oov_tokenizer_char.csv

